In [11]:
import geopandas as gpd
import pandas as pd
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ee
import xarray as xr
import xee
from shapely.geometry import mapping
import json
import warnings
warnings.filterwarnings("ignore")
ee.Authenticate()
import geemap
ee.Initialize( opt_url='https://earthengine-highvolume.googleapis.com')
import matplotlib.image as mpimg
import requests
from io import BytesIO
import numpy as np
import ipywidgets as widgets
from IPython.display import display

## The Brahmaputra


The Brahmaputra is one of the world's major transboundary rivers, flowing through Tibet (China), India, and Bangladesh, and is characterized by an exceptionally dynamic channel morphology — extensive braiding, seasonal channel migration, and some of the highest sediment loads of any river system globally. This morphological instability, combined with intense monsoon-driven discharge variability, makes remote-sensing-based water extent monitoring particularly valuable.

In [2]:

gdf = gpd.read_file('D:/Gokul_R_Kamath/tempp/test/brahmaputra.shp')

In [3]:
gdf=gdf.to_crs("EPSG:4326")

In [4]:
gdf_4326 = gdf.to_crs('EPSG:4326')
aoi_fc = geemap.geopandas_to_ee(gdf_4326)
aoi_geom = aoi_fc.geometry() 

In [5]:
gdf

,objectid,rivname,rilcode,origin,major_trib,bacode,ba_name,sub_basin,shape_Leng,geometry
0,3796,Brahmaputra,1,None,"Dihang/Siang, Lohit/ Tellu R, Subansiri, Kopil...",2B,Brahamaputra,None,2043762.154,"POLYGON ((89.75685 25.75713, 89.7568 25.76567,..."


In [6]:
bbox=gdf.total_bounds.tolist()

In [7]:
bbox

[89.74528132869843, 25.623125345478517, 95.16138267935914, 27.678350930622365]

In [8]:
# converting to gee readable geometry
bbox = ee.Geometry.Rectangle(bbox)


### Setup
## NDWI Yearly Composite Pipeline — Sentinel-2 Surface Water Detection

This notebook derives annual water-extent indicators from Sentinel-2 Surface 
Reflectance imagery using the Normalized Difference Water Index (NDWI), with 
per-pixel cloud/shadow masking via Cloud Score+. For each year, three 
complementary layers are produced:

1. **NDWI yearly median** — a continuous water-likelihood surface
2. **Water occurrence frequency** — fraction of clear-sky observations classified as water
3. **Persistent water mask** — a binary layer thresholded from (2), representing water bodies present through most of the year (as opposed to transient wet surfaces, e.g. post-monsoon saturated soil or paddy fields)

This mirrors the annual-composite structure used elsewhere in the toolkit 
(e.g. ESRI LULC change detection, IMD/GPM rainfall analysis), so outputs are 
directly comparable across years and stackable into a multi-temporal xarray 
dataset for trend analysis.

### 1. Study period and AOI

`years` defines the analysis window (2019–2024 here; adjust to the study 
period). Two AOI representations are used downstream for different purposes:

- **`bbox`** — a rectangular `ee.Geometry` derived from the GeoDataFrame's 
  bounding box (`gdf.total_bounds`). Used for `.filterBounds()` calls, since 
  filtering an ImageCollection against a simple rectangle is computationally 
  cheaper on the GEE server than against a complex polygon, and avoids 
  incorrectly excluding tiles that only partially overlap the true AOI.
- **`aoi_geom`** — the actual (possibly irregular) polygon geometry, 
  converted via `geemap.geopandas_to_ee()`. Used only at the final `.clip()` 
  step, so output rasters are masked to the true boundary rather than the 
  bounding rectangle.

This bbox-for-filtering / polygon-for-clipping split is standard practice in 
GEE workflows to balance query efficiency against spatial precision 
(Gorelick et al., 2017).
### 2. Cloud/shadow masking parameters

- **`QA_BAND = 'cs'`** — the Cloud Score+ quality band, a per-pixel 
  clear-sky probability score (0–1) produced by Google's Cloud Score+ 
  model, which improves on the legacy QA60/SCL cloud masks by explicitly 
  modeling cloud, cloud shadow, cirrus, and haze contamination at the pixel 
  level (Pasquarella et al., 2023; Google Earth Engine, n.d.).
- **`CS_THRESHOLD = 0.6`** — pixels with a clear-sky score below this are 
  masked out. 0.6 is the commonly recommended operational threshold 
  balancing cloud omission against valid-pixel retention; can be tuned 
  stricter (e.g. 0.75) if downstream NDWI results show residual cloud 
  contamination.
### 3. Index definitions

- **NDVI** = (NIR − Red) / (NIR + Red) = (B8 − B4) / (B8 + B4)  
  Standard vegetation index (Rouse et al., 1974), retained here for 
  cross-reference with the LULC/vegetation notebooks in the toolkit.

- **NDWI** = (Green − NIR) / (Green + NIR) = (B3 − B8) / (B3 + B8)  
  McFeeters' (1996) water index, which exploits water's strong NIR 
  absorption and relatively higher green reflectance to separate open 
  water from vegetation and soil. This is distinct from the Gao (1996) 
  NDWI formulation (which uses NIR/SWIR and targets vegetation water 
  content, not open water) — worth noting explicitly in the methods 
  section to avoid ambiguity, since "NDWI" is used inconsistently across 
  the remote sensing literature for two different indices.

- **`NDWI_THRESHOLD = 0.0`** — the standard McFeeters cutoff separating 
  water (NDWI ≥ 0) from non-water. In mountainous terrain, topographic 
  shadow can produce false positives near this threshold (Ji et al., 2009); 
  visual QC against known river/lake extents is recommended before 
  finalizing.

### 4. Per-image masking and index calculation — `mask_and_add_indices()`

Applied via `.map()` across the Sentinel-2 collection (server-side, 
per-image). For each image:

1. The Cloud Score+ band is thresholded to build a binary clear-sky mask.
2. The mask is applied to the image (`updateMask`), so subsequent index 
   calculations only use clear pixels.
3. NDVI and NDWI are computed from the masked reflectance bands.
4. A binary water flag (`NDWI_bin`) is derived per-pixel by thresholding 
   NDWI, with the clear-sky mask re-applied to ensure masked pixels remain 
   masked (rather than defaulting to 0/"not water") through the boolean 
   `.gte()` operation.
5. `system:time_start` is explicitly propagated, since GEE band-math 
   operations do not automatically retain image metadata.
### 5. Yearly composite construction — `make_yearly_ndwi()`

For each year:

1. **Collection filtering**: Sentinel-2 SR Harmonized images are filtered 
   by date, spatial bounds (`bbox`), and a scene-level cloud cover cutoff 
   (`CLOUDY_PIXEL_PERCENTAGE < 80`) as a coarse pre-filter, ahead of the 
   finer per-pixel Cloud Score+ masking.
2. **Cloud Score+ linkage**: the Cloud Score+ collection is joined to the 
   S2 collection via `linkCollection()`, attaching the `cs` band to each 
   matching S2 image.
3. **Per-image masking/indexing**: `mask_and_add_indices()` is mapped 
   across the linked collection.
4. **Temporal reduction**:
   - `NDWI_yearly_median` — pixel-wise median of NDWI across all clear 
     observations in the year. Median (rather than mean) reduces 
     sensitivity to residual cloud/shadow noise and outlier reflectance 
     values (Chastain et al., 2019).
   - `NDWI_water_frequency` — pixel-wise mean of the binary water flag, 
     yielding the fraction of clear observations in which each pixel was 
     classified as water. This is conceptually equivalent to the water 
     occurrence approach used in the JRC Global Surface Water dataset 
     (Pekel et al., 2016), applied here at annual rather than multi-decadal 
     scale.
   - `NDWI_yearly_binary` — a persistent-water mask, thresholding the 
     frequency layer at `PERSISTENCE_THRESHOLD` (default 0.5, i.e. water 
     in >50% of clear observations). This distinguishes stable water 
     bodies from transient/seasonal wet surfaces.
5. **Clipping and metadata**: the three bands are combined into one 
   multi-band image, clipped to the true AOI polygon (`aoi_geom`), and 
   tagged with `year` and `system:time_start` properties for later 
   filtering/sorting.

## 6. Building the year lookup and collection

`ndwi_by_year` is a Python dictionary (`{year: ee.Image}`) built by calling 
`make_yearly_ndwi()` once per year. This is used for:
- fast client-side lookups when plotting or building interactive widgets 
  (e.g. a year-selection dropdown in geemap), and
- constructing `ndwi_yearly_col`, an `ee.ImageCollection` assembled via 
  `ee.ImageCollection.fromImages()`, used for exports and multi-temporal 
  analysis (e.g. via `xee` → `xarray`).

Note that `ee.Image` and `ee.ImageCollection` objects are computed 
**server-side and lazily** — no actual pixel data is transferred until an 
explicit action (`.getInfo()`, `.getThumbURL()`, an export call, or an 
`xee`-based `xr.open_dataset().compute()`) triggers evaluation.
## References

- Chastain, R., Housman, I., Goldstein, J., Finco, M., & Tenneson, K. (2019). 
  Empirical cross sensor comparison of Sentinel-2A and 2B MSI, Landsat-8 OLI, 
  and Landsat-7 ETM+ top of atmosphere spectral characteristics over the 
  conterminous United States. *Remote Sensing of Environment*, 221, 274–285.
- Gao, B.-C. (1996). NDWI—A normalized difference water index for remote 
  sensing of vegetation liquid water from space. *Remote Sensing of 
  Environment*, 58(3), 257–266.
- Google Earth Engine (n.d.). Cloud Score+ (Google/Cloud_Score_Plus/V1). 
  Google Earth Engine Data Catalog.
- Gorelick, N., Hancher, M., Dixon, M., Ilyushchenko, S., Thau, D., & 
  Moore, R. (2017). Google Earth Engine: Planetary-scale geospatial 
  analysis for everyone. *Remote Sensing of Environment*, 202, 18–27.
- Ji, L., Zhang, L., & Wylie, B. (2009). Analysis of dynamic thresholds for 
  the normalized difference water index. *Photogrammetric Engineering & 
  Remote Sensing*, 75(11), 1307–1317.
- McFeeters, S. K. (1996). The use of the Normalized Difference Water Index 
  (NDWI) in the delineation of open water features. *International Journal 
  of Remote Sensing*, 17(7), 1425–1432.
- Pasquarella, V. J., et al. (2023). Cloud Score+: A cloud, cloud shadow, 
  and cirrus/haze masking algorithm for Sentinel-2. Google Research.
- Pekel, J.-F., Cottam, A., Gorelick, N., & Belward, A. S. (2016). 
  High-resolution mapping of global surface water and its long-term 
  changes. *Nature*, 540(7633), 418–422.
- Rouse, J. W., Haas, R. H., Schell, J. A., & Deering, D. W. (1974). 
  Monitoring vegetation systems in the Great Plains with ERTS. *NASA 
  Special Publication*, 351, 309.

In [9]:


years = list(range(2019, 2025))   # adjust to your study period

QA_BAND = 'cs'
CS_THRESHOLD = 0.6

NDWI_THRESHOLD = 0.0
PERSISTENCE_THRESHOLD = 0.5

# =========================================================
# Per-image masking + index calculation
# =========================================================
def mask_and_add_indices(img):
    clear_mask = img.select(QA_BAND).gte(CS_THRESHOLD)
    img_masked = img.updateMask(clear_mask)

    ndvi = img_masked.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = img_masked.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndwi_bin = ndwi.gte(NDWI_THRESHOLD).rename('NDWI_bin').updateMask(clear_mask)

    return ndvi.addBands(ndwi).addBands(ndwi_bin) \
                .set('system:time_start', img.get('system:time_start'))

# =========================================================
# Build yearly composite for a single year
# =========================================================
def make_yearly_ndwi(year):
    time_start = ee.Date.fromYMD(year, 1, 1)
    time_end   = ee.Date.fromYMD(year, 12, 31)

    s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterDate(time_start, time_end)
              .filterBounds(bbox)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
              .select(['B3', 'B4', 'B8']))

    csplus_col = (ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
                  .filterDate(time_start, time_end)
                  .filterBounds(bbox))

    s2_col_linked = s2_col.linkCollection(csplus_col, [QA_BAND])
    s2_indices_col = s2_col_linked.map(mask_and_add_indices)

    ndwi_yearly_median = (s2_indices_col.select('NDWI')
                           .median()
                           .rename('NDWI_yearly_median'))

    ndwi_yearly_freq = (s2_indices_col.select('NDWI_bin')
                         .mean()
                         .rename('NDWI_water_frequency'))

    ndwi_yearly_binary = (ndwi_yearly_freq.gte(PERSISTENCE_THRESHOLD)
                           .rename('NDWI_yearly_binary'))

    ndwi_yearly_raster = (ndwi_yearly_median
                           .addBands(ndwi_yearly_freq)
                           .addBands(ndwi_yearly_binary)
                           .clip(aoi_geom)
                           .set('year', year)
                           .set('system:time_start', time_start.millis()))

    return ndwi_yearly_raster

# =========================================================
# Build dict {year: ee.Image} — used by both geemap dropdown and subplots
# =========================================================
ndwi_by_year = {y: make_yearly_ndwi(y) for y in years}

# Optional: also build the ImageCollection version (for xee / exports later)
ndwi_yearly_col = ee.ImageCollection.fromImages(list(ndwi_by_year.values()))
print('Number of yearly composites:', ndwi_yearly_col.size().getInfo())



Number of yearly composites: 6


In [12]:


# =========================================================
# Visualization params (shared across years for consistent comparison)
# =========================================================
ndwi_vis = {
    'bands': ['NDWI_yearly_median'],
    'min': -0.5,
    'max': 0.5,
    'palette': ['brown', 'white', 'blue']
}
freq_vis = {
    'bands': ['NDWI_water_frequency'],
    'min': 0,
    'max': 1,
    'palette': ['white', 'cyan', 'blue']
}
binary_vis = {
    'bands': ['NDWI_yearly_binary'],
    'min': 0,
    'max': 1,
    'palette': ['white', 'darkblue']
}

vis_map  = {'median': ndwi_vis, 'frequency': freq_vis, 'binary': binary_vis}
band_map = {'median': 'NDWI_yearly_median', 'frequency': 'NDWI_water_frequency', 'binary': 'NDWI_yearly_binary'}

# =========================================================
#  Interactive geemap with year + layer dropdown
# =========================================================


Map = geemap.Map()
Map.centerObject(aoi_geom, zoom=11)
Map.addLayer(aoi_geom, {'color': 'red'}, 'AOI boundary', True, opacity=0.3)

# Fixed layer names — one per band type, always present in the layer panel
LAYER_NAMES = {
    'median': 'NDWI Median',
    'frequency': 'Water Frequency',
    'binary': 'Persistent Water'
}

def update_all_layers(year):
    for key, vis in vis_map.items():
        name = LAYER_NAMES[key]
        img = ndwi_by_year[year].select(vis['bands'])
        if name in Map.ee_layers:
            Map.remove_layer(name)
        Map.addLayer(img, vis, name)

# Initial load — latest year, all three layers added
update_all_layers(years[-1])

year_dropdown = widgets.Dropdown(
    options=years, value=years[-1], description='Year:',
    style={'description_width': 'initial'}
)

def on_year_change(change):
    update_all_layers(year_dropdown.value)

year_dropdown.observe(on_year_change, names='value')

display(year_dropdown, Map)


Dropdown(description='Year:', index=5, options=(2019, 2020, 2021, 2022, 2023, 2024), style=DescriptionStyle(de…

Map(center=[26.56547613921364, 92.48137504406397], controls=(WidgetControl(options=['position', 'transparent_b…